# Module 01: NumPy for Machine Learning
## Notebook 04: Mathematics, Statistics, and Linear Algebra

Machine learning algorithms are fundamentally mathematical models expressed through linear algebra, calculus, and probability. NumPy provides an industrial-grade linear algebra engine (`np.linalg`) and statistical functions essential for training, evaluating, and regularizing models.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Compute axis-wise statistical reductions and utilize `keepdims=True` for seamless broadcasting.
2. Execute vector dot products and matrix multiplications using the `@` operator.
3. Solve linear equations and compute matrix inverses and pseudo-inverses.
4. Calculate eigenvalues and eigenvectors as the foundation for **Principal Component Analysis (PCA)**.
5. Compute $L_1$, $L_2$, and Frobenius norms for regularization penalties (Lasso / Ridge).
6. Generate reproducible synthetic datasets using NumPy's modern `Generator` API (`np.random.default_rng`).

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3


### 1. Statistical Reductions Along Axes & `keepdims`

When working with a 2D data matrix $X$ of shape $(N_{samples}, N_{features})$:
- `axis=0`: Collapses rows (computes statistics **per feature** across all samples).
- `axis=1`: Collapses columns (computes statistics **per sample** across all features).

#### Why `keepdims=True` is Essential:
Without `keepdims=True`, reducing along an axis drops that dimension (e.g., shape `(5, 3)` becomes `(3,)`).
With `keepdims=True`, the reduced dimension is kept as length 1 (shape `(1, 3)` or `(5, 1)`), making broadcasted operations guaranteed to align properly!

In [2]:
# 5 samples, 3 features
X = np.array([
    [10.0, 25.0, 5.0],
    [12.0, 30.0, 8.0],
    [9.0,  20.0, 6.0],
    [15.0, 40.0, 12.0],
    [14.0, 35.0, 9.0]
])

# Feature-wise statistics (axis=0)
mean_feat = np.mean(X, axis=0, keepdims=True)
std_feat = np.std(X, axis=0, keepdims=True)
max_feat = np.max(X, axis=0, keepdims=True)

print(f"Original shape:     {X.shape}")
print(f"Mean shape (keepdims): {mean_feat.shape}")
print("Feature Means:        ", mean_feat)
print("Feature Std Devs:     ", std_feat)

# Guaranteed broadcast: (5, 3) - (1, 3)
X_norm = (X - mean_feat) / std_feat
print("\nNormalized Data:\n", np.round(X_norm, 3))

Original shape:     (5, 3)
Mean shape (keepdims): (1, 3)
Feature Means:         [[12. 30.  8.]]
Feature Std Devs:      [[2.28035085 7.07106781 2.44948974]]

Normalized Data:
 [[-0.877 -0.707 -1.225]
 [ 0.     0.     0.   ]
 [-1.316 -1.414 -0.816]
 [ 1.316  1.414  1.633]
 [ 0.877  0.707  0.408]]


---
### 2. Matrix Multiplication and Dot Products

> **Crucial Rule:**
> - `*` performs **element-wise** multiplication (Hadamard product).
> - `@` (or `np.matmul()`) performs **true matrix multiplication** ($C_{ik} = \sum_j A_{ij} B_{jk}$).

In [3]:
# Vector Dot Product: u . v = \sum u_i * v_i
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

dot_product = np.dot(u, v) # or u @ v
print(f"Dot product (1*4 + 2*5 + 3*6): {dot_product}")

# Matrix-Vector Multiplication: y_pred = X @ w
# 3 samples, 2 features
X_mat = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0]
])
# Weight vector: 2 features -> 1 output
w = np.array([0.5, -0.2])

predictions = X_mat @ w
print("\nFeature Matrix X (3x2):\n", X_mat)
print("Weights w (2,):", w)
print("Predictions X @ w (3,):", predictions)

# Matrix Transposition
print("\nTransposed X (2x3):\n", X_mat.T)

Dot product (1*4 + 2*5 + 3*6): 32.0

Feature Matrix X (3x2):
 [[1. 2.]
 [3. 4.]
 [5. 6.]]
Weights w (2,): [ 0.5 -0.2]
Predictions X @ w (3,): [0.1 0.7 1.3]

Transposed X (2x3):
 [[1. 3. 5.]
 [2. 4. 6.]]


---
### 3. Linear Algebra Submodule: `np.linalg`

NumPy's `np.linalg` wraps optimized BLAS and LAPACK routines.

#### Key Functions:
- `np.linalg.det(A)`: Computes matrix determinant.
- `np.linalg.inv(A)`: Computes matrix inverse $A^{-1}$ ($A A^{-1} = I$).
- `np.linalg.pinv(A)`: Computes Moore-Penrose pseudo-inverse (works even if matrix is singular or non-square!).
- `np.linalg.solve(A, b)`: Solves linear system $A x = b$ (more stable and faster than `inv(A) @ b`).

In [4]:
# Solving a linear system:
# 2x + 3y = 8
# 4x + 9y = 20
A = np.array([[2.0, 3.0], [4.0, 9.0]])
b = np.array([8.0, 20.0])

det_A = np.linalg.det(A)
print(f"Determinant of A: {det_A:.2f}")

# Solve Ax = b
solution = np.linalg.solve(A, b)
print(f"Solution [x, y]:  {solution}")

# Verify solution: A @ solution == b
print("Verification A @ x:", A @ solution)

Determinant of A: 6.00
Solution [x, y]:  [2.         1.33333333]
Verification A @ x: [ 8. 20.]


---
### 4. Eigenvalues and Eigenvectors (PCA Foundation)

For a square matrix $A$, a non-zero vector $v$ is an eigenvector with eigenvalue $\lambda$ if:
$$A v = \lambda v$$

In Machine Learning, computing the eigenvectors of the **Covariance Matrix** of your data reveals the **Principal Components**—the directions of maximal variance!

In [5]:
# Symmetric Covariance Matrix
cov_matrix = np.array([
    [2.5, 0.8],
    [0.8, 1.2]
])

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

print("Covariance Matrix:\n", cov_matrix)
print("\nEigenvalues:     ", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

# Percentage of total variance explained by each principal axis
explained_var_ratio = eigenvalues / np.sum(eigenvalues)
print("\nVariance explained ratio:", np.round(explained_var_ratio, 4))
print(f"First Principal Component captures {explained_var_ratio[-1]*100:.1f}% of variance!")

Covariance Matrix:
 [[2.5 0.8]
 [0.8 1.2]]

Eigenvalues:      [0.81922359 2.88077641]
Eigenvectors (columns):
 [[ 0.42977167 -0.9029376 ]
 [-0.9029376  -0.42977167]]

Variance explained ratio: [0.2214 0.7786]
First Principal Component captures 77.9% of variance!


---
### 5. Vector and Matrix Norms (Regularization)

Norms quantify the "magnitude" of vectors and matrices, forming the mathematical backbone of regularization:
- **$L_1$ Norm (Manhattan / Lasso penalty)**: $\|w\|_1 = \sum |w_i|$ (encourages sparsity/feature selection).
- **$L_2$ Norm (Euclidean / Ridge penalty)**: $\|w\|_2 = \sqrt{\sum w_i^2}$ (shrinks weights toward zero).
- **Frobenius Norm**: Matrix equivalent of $L_2$ norm.

In [6]:
w_weights = np.array([3.0, -4.0, 0.0, 1.5, -0.5])

# L1 Norm (Lasso penalty)
l1_norm = np.linalg.norm(w_weights, ord=1)
print(f"L1 Norm (Sum of absolute values): {l1_norm}")

# L2 Norm (Euclidean length / Ridge penalty)
l2_norm = np.linalg.norm(w_weights, ord=2)
print(f"L2 Norm (Euclidean magnitude):    {l2_norm:.4f}")

# Frobenius norm of a weight matrix
W_matrix = np.array([[1.0, 2.0], [3.0, 4.0]])
frobenius_norm = np.linalg.norm(W_matrix, ord='fro')
print(f"Frobenius Norm of matrix:         {frobenius_norm:.4f}")

L1 Norm (Sum of absolute values): 9.0
L2 Norm (Euclidean magnitude):    5.2440
Frobenius Norm of matrix:         5.4772


---
### 6. Modern Random Number Generation (`default_rng`)

> **NumPy Modern Practice:**
> Avoid legacy `np.random.seed()` or `np.random.randn()`.
> Always instantiate a `Generator` via `np.random.default_rng(seed)`.
> It uses the high-performance **PCG64** bit generator, has superior statistical properties, and is thread-safe!

In [7]:
# 1. Instantiate the generator with a fixed seed for reproducible experiments
rng = np.random.default_rng(seed=42)

# 2. Gaussian / Normal distribution: N(mean=0, std=1)
# Simulating weight initialization for a neural layer (3 inputs, 4 neurons)
weights = rng.normal(loc=0.0, scale=0.1, size=(3, 4))
print("Gaussian Weights (3x4):\n", np.round(weights, 4))

# 3. Uniform distribution: U(low=0.0, high=1.0)
uniform_noise = rng.uniform(low=0.0, high=1.0, size=5)
print("\nUniform random values:", np.round(uniform_noise, 4))

# 4. Random permutation for train/test splitting
indices = rng.permutation(10)
print("\nShuffled indices of 10 samples:", indices)

Gaussian Weights (3x4):
 [[ 0.0305 -0.104   0.075   0.0941]
 [-0.1951 -0.1302  0.0128 -0.0316]
 [-0.0017 -0.0853  0.0879  0.0778]]

Uniform random values: [0.6439 0.8228 0.4434 0.2272 0.5546]

Shuffled indices of 10 samples: [6 1 2 7 9 5 8 4 0 3]


### Summary & Next Steps
In this notebook, you mastered:
- Computing axis-wise statistical reductions with `keepdims=True`.
- Vector dot products and matrix multiplications (`@`).
- Matrix determinants, solving linear systems, and inversions.
- Computing eigenvectors and understanding variance explained in PCA.
- Regularization norms ($L_1$, $L_2$, Frobenius).
- Modern random generation with `np.random.default_rng`.

**Next Notebook:** `05_practical_ml_applications.ipynb` — Implement end-to-end machine learning algorithms (Linear Regression with Gradient Descent, Logistic Regression, and K-Nearest Neighbors) from scratch using pure NumPy!